# Silver transformation and data quality

This notebook orchestrates the reusable Bronze and Silver modules, reports data quality outcomes, and validates the local sample results. All datasets are processed as in-memory Spark DataFrames; no Delta/Parquet tables are written locally.

In [ ]:
from pathlib import Path
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if not (project_root / "config" / "bronze_sources.json").is_file():
    raise FileNotFoundError(f"Project root not found: {project_root}")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.bronze import load_bronze_datasets
from src.silver import build_silver_datasets

spark = (
    SparkSession.builder
    .appName("InsuranceLakehouseSilverTransformation")
    .master("local[*]")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Project root: {project_root}")

In [ ]:
# Run the reusable in-memory pipeline
bronze_dfs = load_bronze_datasets(spark, project_root)
silver_valid_dfs, silver_rejected_dfs = build_silver_datasets(
    bronze_dfs, project_root
)

print(f"Bronze datasets available: {len(bronze_dfs)}")
print(f"Silver datasets validated: {len(silver_valid_dfs)}")

In [ ]:
# Quality summary and checks for the synthetic sample
expected_counts = {
    "customers": {"valid": 8, "rejected": 1},
    "policies": {"valid": 8, "rejected": 2},
    "claims": {"valid": 7, "rejected": 2},
}

print("FINAL SILVER QUALITY SUMMARY")
print("-" * 65)

total_valid = 0
total_rejected = 0

for dataset_name in bronze_dfs:
    valid = silver_valid_dfs[dataset_name].count()
    rejected = silver_rejected_dfs[dataset_name].count()
    total = valid + rejected
    total_valid += valid
    total_rejected += rejected

    print(
        f"{dataset_name:<10} | total={total:>3} | "
        f"valid={valid:>3} | rejected={rejected:>3}"
    )

    if dataset_name in expected_counts:
        expected = expected_counts[dataset_name]
        assert valid == expected["valid"], (
            f"Unexpected valid count for {dataset_name}: {valid}"
        )
        assert rejected == expected["rejected"], (
            f"Unexpected rejected count for {dataset_name}: {rejected}"
        )

print("-" * 65)
print(f"Valid records    : {total_valid}")
print(f"Rejected records : {total_rejected}")

assert total_valid == 23, f"Unexpected valid total: {total_valid}"
assert total_rejected == 5, f"Unexpected rejected total: {total_rejected}"
print("PASS | Silver sample counts validated")

In [ ]:
# Explain every rejection without displaying customer details
print("REJECTION REASONS")
print("-" * 65)

for dataset_name, rejected_df in silver_rejected_dfs.items():
    print(f"\n{dataset_name.upper()}")

    rejection_summary = (
        rejected_df
        .select(F.explode("_dq_errors").alias("reason"))
        .groupBy("reason")
        .count()
        .orderBy("reason")
        .collect()
    )

    if not rejection_summary:
        print("No rejected records")

    for row in rejection_summary:
        print(f"{row['reason']:<45} | {row['count']}")